# Project 3

In [1]:
import csv
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd
from scipy.stats import norm

## Primary task

In [2]:
std = np.sqrt(np.log(2))

arrival_a = lambda t: -(1/3650) * t**2 + (1/10) * t
arrival_b = lambda t: (1/5) * arrival_a(t)
arrival_c = lambda t: 6.0 
mu_a = np.log(4 * np.sqrt(2))
mu_b = np.log(6 * np.sqrt(2))
mu_c = np.log(5 * np.sqrt(2))

In [3]:
class Ward:
    def __init__(self, name, arrival_func, u, std, beds=None):
        self.name = name
        self.arrival_func = arrival_func 
        self.u = u
        self.std = std
        self.beds = beds

    def arrival_rate(self, t):
        return self.arrival_func(t)

    def length_of_stay(self):
        return np.random.lognormal(self.u, self.std)
    
def get_all_bed_distributions(total_beds=75):
    distributions = []
    for beds_a in range(1, total_beds - 1):
        for beds_b in range(1, total_beds - beds_a):
            beds_c = total_beds - beds_a - beds_b
            distributions.append((beds_a, beds_b, beds_c))
    return distributions

def simulate_system(ward_a, ward_b, ward_c, n_days=365):
    occupancy_a = []
    occupancy_b = []
    occupancy_c = []
    
    relocated_a = 0
    relocated_b = 0
    relocated_c = 0
    
    full_on_arrival_a = 0
    full_on_arrival_b = 0
    full_on_arrival_c = 0
    
    total_arrivals_a = 0
    total_arrivals_b = 0
    total_arrivals_c = 0
    
    utilization_history_a = []
    utilization_history_b = []
    utilization_history_c = []
    
    for day in range(n_days):
        occupancy_a = [d for d in occupancy_a if d > day]
        occupancy_b = [d for d in occupancy_b if d > day]
        occupancy_c = [d for d in occupancy_c if d > day]
        
        arrivals_a = np.random.poisson(ward_a.arrival_rate(day))
        arrivals_b = np.random.poisson(ward_b.arrival_rate(day))
        arrivals_c = np.random.poisson(ward_c.arrival_rate(day))
        
        total_arrivals_a += arrivals_a
        total_arrivals_b += arrivals_b
        total_arrivals_c += arrivals_c
        
        for _ in range(arrivals_b):
            if len(occupancy_b) >= ward_b.beds:
                full_on_arrival_b += 1
                
            if len(occupancy_b) < ward_b.beds:
                occupancy_b.append(day + ward_b.length_of_stay())
            elif len(occupancy_a) < ward_a.beds:
                occupancy_a.append(day + ward_b.length_of_stay())
            else:
                relocated_b += 1
                
        for _ in range(arrivals_a):
            if len(occupancy_a) >= ward_a.beds:
                full_on_arrival_a += 1
                
            if len(occupancy_a) < ward_a.beds:
                occupancy_a.append(day + ward_a.length_of_stay())
            else:
                relocated_a += 1
                
        for _ in range(arrivals_c):
            if len(occupancy_c) >= ward_c.beds:
                full_on_arrival_c += 1
                
            if len(occupancy_c) < ward_c.beds:
                occupancy_c.append(day + ward_c.length_of_stay())
            else:
                relocated_c += 1
                
        utilization_history_a.append(len(occupancy_a))
        utilization_history_b.append(len(occupancy_b))
        utilization_history_c.append(len(occupancy_c))
                
    total_relocated = relocated_a + relocated_b + relocated_c
    
    prob_full_a = full_on_arrival_a / total_arrivals_a if total_arrivals_a > 0 else 0.0
    prob_full_b = full_on_arrival_b / total_arrivals_b if total_arrivals_b > 0 else 0.0
    prob_full_c = full_on_arrival_c / total_arrivals_c if total_arrivals_c > 0 else 0.0
    
    mean_util_a = (np.mean(utilization_history_a) / ward_a.beds) if ward_a.beds > 0 else 0.0
    mean_util_b = (np.mean(utilization_history_b) / ward_b.beds) if ward_b.beds > 0 else 0.0
    mean_util_c = (np.mean(utilization_history_c) / ward_c.beds) if ward_c.beds > 0 else 0.0
    
    return {
    "relocations": {
        "A": relocated_a,
        "B": relocated_b,
        "C": relocated_c,
        "Total": total_relocated
    },
    "prob_full_on_arrival": {
        "A": prob_full_a,
        "B": prob_full_b,
        "C": prob_full_c
    },
    "mean_utilization": {
        "A": mean_util_a,
        "B": mean_util_b,
        "C": mean_util_c
    },
    "history": [
        utilization_history_a,
        utilization_history_b,
        utilization_history_c
    ]
}

def find_optimal_distribution(distributions, ward_a, ward_b, ward_c, replications=3):
    best_distribution = None
    min_avg_relocated = float('inf')
    
    total_configs = len(distributions)
    print(f"Testing {total_configs} bed distributions...")
    
    for i, (beds_a, beds_b, beds_c) in enumerate(distributions):
        ward_a.beds = beds_a
        ward_b.beds = beds_b
        ward_c.beds = beds_c
        
        total_relocations_sum = 0
        
        
        # Run multiple replications to smooth stochastic variance
        for _ in range(replications):
            results = simulate_system(ward_a, ward_b, ward_c, n_days=365)
            total_relocations_sum += results["relocations"]["Total"]

            
        avg_relocated = total_relocations_sum / replications

        # write results to CSV for analysis
        with open('bed_distribution_results.csv', mode='a', newline='') as file:
            writer = csv.writer(file)
            writer.writerow([beds_a, beds_b, beds_c, avg_relocated])
        
        if avg_relocated < min_avg_relocated:
            min_avg_relocated = avg_relocated
            best_distribution = (beds_a, beds_b, beds_c)
            
        if (i + 1) % 500 == 0:
            print(f"Processed {i + 1}/{total_configs} configurations...")
            
    return best_distribution, min_avg_relocated

In [ ]:
total_beds = 75
dist = get_all_bed_distributions(total_beds=total_beds)
ward_a = Ward("Ward A", arrival_func=arrival_a, u=mu_a, std=std, beds=0)
ward_b = Ward("Ward B", arrival_func=arrival_b, u=mu_b, std=std, beds=0)
ward_c = Ward("Ward C", arrival_func=arrival_c, u=mu_c, std=std, beds=0)

optimal_beds, min_relocated = find_optimal_distribution(dist, ward_a, ward_b, ward_c, replications=1)

print("\n--- OPTIMAL CONFIGURATION FOUND ---")
print(f"Ward A Beds: {optimal_beds[0]}")
print(f"Ward B Beds: {optimal_beds[1]}")
print(f"Ward C Beds: {optimal_beds[2]}")
print(f"Minimum Average Relocated Patients: {min_relocated:.2f}")

# To view the full statistics for the winning configuration, you can run it once more:
ward_a.beds, ward_b.beds, ward_c.beds = optimal_beds
np.random.seed(42) 
final_results = simulate_system(ward_a, ward_b, ward_c, n_days=365)
print("\nFinal Results for Optimal Distribution:")
print(final_results)

## Primary performance measures

In [ ]:
def run_replications(wards, n_days, R):
    results = []

    for r in range(R):
        # vigtigt: ny seed eller ingen seed her, så replications varierer
        res = simulate_system(*wards, n_days=n_days)
        results.append(res)

    return results

def ci(x):
    x = np.array(x)
    m = x.mean()
    h = norm.ppf(.975) * x.std(ddof=1) / np.sqrt(len(x))
    return m, m-h, m+h

def show(title, data):
    print(f"\n--- {title} ---")
    for name, values in data.items():
        m, l, u = ci(values)
        print(f"{name}: {m:.3f} (95% CI: [{l:.3f}, {u:.3f}])")

In [ ]:
random_seed = 42
np.random.seed(random_seed)

n = 365
wards = [
    Ward("Ward A", arrival_func=arrival_a, u=mu_a, std=std, beds=0),
    Ward("Ward B", arrival_func=arrival_b, u=mu_b, std=std, beds=0),
    Ward("Ward C", arrival_func=arrival_c, u=mu_c, std=std, beds=0)
]

for ward, beds in zip(wards, optimal_beds):
    ward.beds = beds

R = 500
results = run_replications(wards, n_days=n, R=R)

pfull = {k: [] for k in "ABC"}
rel = {k: [] for k in ["A", "B", "C", "Total"]}
util = {k: [] for k in "ABC"}

for res in results:
    for k, v in res["prob_full_on_arrival"].items():
        pfull[k].append(v)

    for k, v in res["relocations"].items():
        rel[k].append(v)

    for k, v in res["mean_utilization"].items():
        util[k].append(v)


show("Probability full on arrival", {
    "Ward A": pfull["A"],
    "Ward B": pfull["B"],
    "Ward C": pfull["C"]
})

show("Relocations", {
    "Ward A": rel["A"],
    "Ward B": rel["B"],
    "Ward C": rel["C"],
    "Total": rel["Total"]
})

show("Mean utilization", {
    "Ward A": util["A"],
    "Ward B": util["B"],
    "Ward C": util["C"]
})

In [ ]:
np.random.seed(42)
final = simulate_system(*wards, n_days=n)
history = final["history"]

df_history = pd.DataFrame({
    "A": history[0],
    "B": history[1],
    "C": history[2]
})

plt.figure(figsize=(12,6))
plt.plot(df_history["A"], label="Ward A")
plt.plot(df_history["B"], label="Ward B")
plt.plot(df_history["C"], label="Ward C")
plt.xlabel("Day")
plt.ylabel("Occupied beds")
plt.title("Daily Occupancy for Optimal Bed Distribution")
plt.legend()
plt.grid(True)
plt.show()

plt.figure(figsize=(12,6))
plt.plot(arrival_a(np.arange(n)), label="Arrival Rate A")
plt.plot(arrival_b(np.arange(n)), label="Arrival Rate B")
plt.plot(np.arange(n), np.full(n, arrival_c(0)), label="Arrival Rate C")
plt.xlabel("Day")
plt.ylabel("Arrival Rate")
plt.title("Daily Arrival Rates for Each Ward")
plt.legend()
plt.grid(True)
plt.show()

## Sensitivity analysis

### Lognormal distribution

In [ ]:
import csv
import numpy as np

class Ward:
    def __init__(self, name, arrival_func, u, std, mean_los=1, beds=None, use_exponential=True):
        self.name = name
        self.arrival_func = arrival_func 
        self.u = u
        self.std = std
        self.mean_los = mean_los
        self.use_exponential = use_exponential
        self.beds = beds

    def arrival_rate(self, t):
        return self.arrival_func(t)

    def length_of_stay(self, rng):
        if self.use_exponential:
            return rng.exponential(self.mean_los)
        else:
            return rng.lognormal(self.u, self.std)
    
common_std = np.sqrt(np.log(2))

arrival_a = lambda t: -(1/3650) * t**2 + (1/10) * t
arrival_b = lambda t: (1/5) * arrival_a(t)
arrival_c = lambda t: 6.0 
mu_a = np.log(4 * np.sqrt(2))
mu_b = np.log(6 * np.sqrt(2))
mu_c = np.log(5 * np.sqrt(2))
a_scale = 8
b_scale = 12
c_scale = 10

def get_all_bed_distributions(total_beds=75):
    distributions = []
    for beds_a in range(1, total_beds - 1):
        for beds_b in range(1, total_beds - beds_a):
            beds_c = total_beds - beds_a - beds_b
            distributions.append((total_beds, beds_a, beds_b, beds_c))
    return distributions

def simulate_system(ward_a, ward_b, ward_c, rng, n_days=365):
    occupancy_a = []
    occupancy_b = []
    occupancy_c = []
    
    relocated_a = 0
    relocated_b = 0
    relocated_c = 0
    
    full_on_arrival_a = 0
    full_on_arrival_b = 0
    full_on_arrival_c = 0
    
    total_arrivals_a = 0
    total_arrivals_b = 0
    total_arrivals_c = 0
    
    utilization_history_a = []
    utilization_history_b = []
    utilization_history_c = []
    
    for day in range(n_days):
        occupancy_a = [d for d in occupancy_a if d > day]
        occupancy_b = [d for d in occupancy_b if d > day]
        occupancy_c = [d for d in occupancy_c if d > day]
        
        arrivals_a = rng.poisson(ward_a.arrival_rate(day))
        arrivals_b = rng.poisson(ward_b.arrival_rate(day))
        arrivals_c = rng.poisson(ward_c.arrival_rate(day))
        
        total_arrivals_a += arrivals_a
        total_arrivals_b += arrivals_b
        total_arrivals_c += arrivals_c
        
        for _ in range(arrivals_b):
            if len(occupancy_b) >= ward_b.beds:
                full_on_arrival_b += 1
                
            if len(occupancy_b) < ward_b.beds:
                occupancy_b.append(day + ward_b.length_of_stay(rng))
            elif len(occupancy_a) < ward_a.beds:
                occupancy_a.append(day + ward_b.length_of_stay(rng))
            else:
                relocated_b += 1
                
        for _ in range(arrivals_a):
            if len(occupancy_a) >= ward_a.beds:
                full_on_arrival_a += 1
                
            if len(occupancy_a) < ward_a.beds:
                occupancy_a.append(day + ward_a.length_of_stay(rng))
            else:
                relocated_a += 1
                
        for _ in range(arrivals_c):
            if len(occupancy_c) >= ward_c.beds:
                full_on_arrival_c += 1
                
            if len(occupancy_c) < ward_c.beds:
                occupancy_c.append(day + ward_c.length_of_stay(rng))
            else:
                relocated_c += 1
                
        utilization_history_a.append(len(occupancy_a))
        utilization_history_b.append(len(occupancy_b))
        utilization_history_c.append(len(occupancy_c))
                
    total_relocated = relocated_a + relocated_b + relocated_c
    
    prob_full_a = full_on_arrival_a / total_arrivals_a if total_arrivals_a > 0 else 0.0
    prob_full_b = full_on_arrival_b / total_arrivals_b if total_arrivals_b > 0 else 0.0
    prob_full_c = full_on_arrival_c / total_arrivals_c if total_arrivals_c > 0 else 0.0
    
    mean_util_a = (np.mean(utilization_history_a) / ward_a.beds) if ward_a.beds > 0 else 0.0
    mean_util_b = (np.mean(utilization_history_b) / ward_b.beds) if ward_b.beds > 0 else 0.0
    mean_util_c = (np.mean(utilization_history_c) / ward_c.beds) if ward_c.beds > 0 else 0.0
    
    return {
        "relocations": {"A": relocated_a, "B": relocated_b, "C": relocated_c, "Total": total_relocated},
        "prob_full_on_arrival": {"A": prob_full_a, "B": prob_full_b, "C": prob_full_c},
        "mean_utilization": {"A": mean_util_a, "B": mean_util_b, "C": mean_util_c}
    }

def find_optimal_distribution(distributions, ward_a, ward_b, ward_c, replications=3, csv_filename='best_bed_distributions.csv'):
    best_distribution = None
    min_avg_relocated = float('inf')
    best_metrics = None
    
    total_configs = len(distributions)
    print(f"Testing {total_configs} bed distributions...")
    
    rep_seeds = [42 + i for i in range(replications)]
    
    for i, (total_beds, beds_a, beds_b, beds_c) in enumerate(distributions):
        ward_a.beds = beds_a
        ward_b.beds = beds_b
        ward_c.beds = beds_c
        
        metrics_sum = {
            "reloc_tot": 0.0,
            "prob_a": 0.0, "prob_b": 0.0, "prob_c": 0.0,
            "util_a": 0.0, "util_b": 0.0, "util_c": 0.0
        }
        
        for rep in range(replications):
            rng = np.random.default_rng(rep_seeds[rep])
            results = simulate_system(ward_a, ward_b, ward_c, rng, n_days=365)
            
            metrics_sum["reloc_tot"] += results["relocations"]["Total"]
            metrics_sum["prob_a"] += results["prob_full_on_arrival"]["A"]
            metrics_sum["prob_b"] += results["prob_full_on_arrival"]["B"]
            metrics_sum["prob_c"] += results["prob_full_on_arrival"]["C"]
            metrics_sum["util_a"] += results["mean_utilization"]["A"]
            metrics_sum["util_b"] += results["mean_utilization"]["B"]
            metrics_sum["util_c"] += results["mean_utilization"]["C"]
            
        avg = {k: v / replications for k, v in metrics_sum.items()}
        
        if avg["reloc_tot"] < min_avg_relocated:
            min_avg_relocated = avg["reloc_tot"]
            best_distribution = (total_beds, beds_a, beds_b, beds_c)
            best_metrics = avg
            
        if (i + 1) % 500 == 0:
            print(f"Processed {i + 1}/{total_configs} configurations...")
            
    with open(csv_filename, mode='a', newline='') as file:
        writer = csv.writer(file)
        writer.writerow([
            best_distribution[0], best_distribution[1], best_distribution[2], best_distribution[3], 
            best_metrics["reloc_tot"], 
            best_metrics["prob_a"], best_metrics["prob_b"], best_metrics["prob_c"], 
            best_metrics["util_a"], best_metrics["util_b"], best_metrics["util_c"]
        ])
            
    return best_distribution, min_avg_relocated


csv_file = 'best_bed_distributions_lognormal.csv'
np.random.seed(42) 
with open(csv_file, mode='w', newline='') as file:
    writer = csv.writer(file)
    writer.writerow([
        'Total_Beds', 'Beds_A', 'Beds_B', 'Beds_C', 
        'Avg_Relocated_Total', 
        'Avg_Prob_Full_A', 'Avg_Prob_Full_B', 'Avg_Prob_Full_C', 
        'Avg_Util_A', 'Avg_Util_B', 'Avg_Util_C'
    ])

ward_a = Ward("Ward A", arrival_func=arrival_a, u=mu_a, mean_los=a_scale, std=common_std, beds=0, use_exponential=False)
ward_b = Ward("Ward B", arrival_func=arrival_b, u=mu_b, mean_los=b_scale, std=common_std, beds=0, use_exponential=False)
ward_c = Ward("Ward C", arrival_func=arrival_c, u=mu_c, mean_los=c_scale, std=common_std, beds=0, use_exponential=False)

scenarios = [25,50, 75, 100, 125]

for total_capacity in scenarios:
    print(f"\n--- RUNNING SCENARIO: {total_capacity} TOTAL BEDS ---")
    dist = get_all_bed_distributions(total_beds=total_capacity)
    
    optimal_beds, min_relocated = find_optimal_distribution(dist, ward_a, ward_b, ward_c, replications=5, csv_filename=csv_file)

    print(f"\nOptimal Configuration for {total_capacity} beds:")
    print(f"Ward A: {optimal_beds[1]}, Ward B: {optimal_beds[2]}, Ward C: {optimal_beds[3]}")
    print(f"Minimum Average Relocated Patients: {min_relocated:.2f}")
    print("-" * 40)

### Exponential distribution

In [ ]:
import csv
import numpy as np

class Ward:
    def __init__(self, name, arrival_func, u, std, mean_los=1, beds=None, use_exponential=True):
        self.name = name
        self.arrival_func = arrival_func 
        self.u = u
        self.std = std
        self.mean_los = mean_los
        self.use_exponential = use_exponential
        self.beds = beds

    def arrival_rate(self, t):
        return self.arrival_func(t)

    def length_of_stay(self, rng):
        if self.use_exponential:
            return rng.exponential(self.mean_los)
        else:
            return rng.lognormal(self.u, self.std)
    
common_std = np.sqrt(np.log(2))

arrival_a = lambda t: -(1/3650) * t**2 + (1/10) * t
arrival_b = lambda t: (1/5) * arrival_a(t)
arrival_c = lambda t: 6.0 
mu_a = np.log(4 * np.sqrt(2))
mu_b = np.log(6 * np.sqrt(2))
mu_c = np.log(5 * np.sqrt(2))
a_scale = 8
b_scale = 12
c_scale = 10

def get_all_bed_distributions(total_beds=75):
    distributions = []
    for beds_a in range(1, total_beds - 1):
        for beds_b in range(1, total_beds - beds_a):
            beds_c = total_beds - beds_a - beds_b
            distributions.append((total_beds, beds_a, beds_b, beds_c))
    return distributions

def simulate_system(ward_a, ward_b, ward_c, rng, n_days=365):
    occupancy_a = []
    occupancy_b = []
    occupancy_c = []
    
    relocated_a = 0
    relocated_b = 0
    relocated_c = 0
    
    full_on_arrival_a = 0
    full_on_arrival_b = 0
    full_on_arrival_c = 0
    
    total_arrivals_a = 0
    total_arrivals_b = 0
    total_arrivals_c = 0
    
    utilization_history_a = []
    utilization_history_b = []
    utilization_history_c = []
    
    for day in range(n_days):
        occupancy_a = [d for d in occupancy_a if d > day]
        occupancy_b = [d for d in occupancy_b if d > day]
        occupancy_c = [d for d in occupancy_c if d > day]
        
        arrivals_a = rng.poisson(ward_a.arrival_rate(day))
        arrivals_b = rng.poisson(ward_b.arrival_rate(day))
        arrivals_c = rng.poisson(ward_c.arrival_rate(day))
        
        total_arrivals_a += arrivals_a
        total_arrivals_b += arrivals_b
        total_arrivals_c += arrivals_c
        
        for _ in range(arrivals_b):
            if len(occupancy_b) >= ward_b.beds:
                full_on_arrival_b += 1
                
            if len(occupancy_b) < ward_b.beds:
                occupancy_b.append(day + ward_b.length_of_stay(rng))
            elif len(occupancy_a) < ward_a.beds:
                occupancy_a.append(day + ward_b.length_of_stay(rng))
            else:
                relocated_b += 1
                
        for _ in range(arrivals_a):
            if len(occupancy_a) >= ward_a.beds:
                full_on_arrival_a += 1
                
            if len(occupancy_a) < ward_a.beds:
                occupancy_a.append(day + ward_a.length_of_stay(rng))
            else:
                relocated_a += 1
                
        for _ in range(arrivals_c):
            if len(occupancy_c) >= ward_c.beds:
                full_on_arrival_c += 1
                
            if len(occupancy_c) < ward_c.beds:
                occupancy_c.append(day + ward_c.length_of_stay(rng))
            else:
                relocated_c += 1
                
        utilization_history_a.append(len(occupancy_a))
        utilization_history_b.append(len(occupancy_b))
        utilization_history_c.append(len(occupancy_c))
                
    total_relocated = relocated_a + relocated_b + relocated_c
    
    prob_full_a = full_on_arrival_a / total_arrivals_a if total_arrivals_a > 0 else 0.0
    prob_full_b = full_on_arrival_b / total_arrivals_b if total_arrivals_b > 0 else 0.0
    prob_full_c = full_on_arrival_c / total_arrivals_c if total_arrivals_c > 0 else 0.0
    
    mean_util_a = (np.mean(utilization_history_a) / ward_a.beds) if ward_a.beds > 0 else 0.0
    mean_util_b = (np.mean(utilization_history_b) / ward_b.beds) if ward_b.beds > 0 else 0.0
    mean_util_c = (np.mean(utilization_history_c) / ward_c.beds) if ward_c.beds > 0 else 0.0
    
    return {
        "relocations": {"A": relocated_a, "B": relocated_b, "C": relocated_c, "Total": total_relocated},
        "prob_full_on_arrival": {"A": prob_full_a, "B": prob_full_b, "C": prob_full_c},
        "mean_utilization": {"A": mean_util_a, "B": mean_util_b, "C": mean_util_c}
    }

def find_optimal_distribution(distributions, ward_a, ward_b, ward_c, replications=3, csv_filename='best_bed_distributions.csv'):
    best_distribution = None
    min_avg_relocated = float('inf')
    best_metrics = None
    
    total_configs = len(distributions)
    print(f"Testing {total_configs} bed distributions...")
    
    rep_seeds = [42 + i for i in range(replications)]
    
    for i, (total_beds, beds_a, beds_b, beds_c) in enumerate(distributions):
        ward_a.beds = beds_a
        ward_b.beds = beds_b
        ward_c.beds = beds_c
        
        metrics_sum = {
            "reloc_tot": 0.0,
            "prob_a": 0.0, "prob_b": 0.0, "prob_c": 0.0,
            "util_a": 0.0, "util_b": 0.0, "util_c": 0.0
        }
        
        for rep in range(replications):
            rng = np.random.default_rng(rep_seeds[rep])
            results = simulate_system(ward_a, ward_b, ward_c, rng, n_days=365)
            
            metrics_sum["reloc_tot"] += results["relocations"]["Total"]
            metrics_sum["prob_a"] += results["prob_full_on_arrival"]["A"]
            metrics_sum["prob_b"] += results["prob_full_on_arrival"]["B"]
            metrics_sum["prob_c"] += results["prob_full_on_arrival"]["C"]
            metrics_sum["util_a"] += results["mean_utilization"]["A"]
            metrics_sum["util_b"] += results["mean_utilization"]["B"]
            metrics_sum["util_c"] += results["mean_utilization"]["C"]
            
        avg = {k: v / replications for k, v in metrics_sum.items()}
        
        if avg["reloc_tot"] < min_avg_relocated:
            min_avg_relocated = avg["reloc_tot"]
            best_distribution = (total_beds, beds_a, beds_b, beds_c)
            best_metrics = avg
            
        if (i + 1) % 500 == 0:
            print(f"Processed {i + 1}/{total_configs} configurations...")
            
    with open(csv_filename, mode='a', newline='') as file:
        writer = csv.writer(file)
        writer.writerow([
            best_distribution[0], best_distribution[1], best_distribution[2], best_distribution[3], 
            best_metrics["reloc_tot"], 
            best_metrics["prob_a"], best_metrics["prob_b"], best_metrics["prob_c"], 
            best_metrics["util_a"], best_metrics["util_b"], best_metrics["util_c"]
        ])
            
    return best_distribution, min_avg_relocated


csv_file = 'best_bed_distributions_exponential.csv'
np.random.seed(42) 
with open(csv_file, mode='w', newline='') as file:
    writer = csv.writer(file)
    writer.writerow([
        'Total_Beds', 'Beds_A', 'Beds_B', 'Beds_C', 
        'Avg_Relocated_Total', 
        'Avg_Prob_Full_A', 'Avg_Prob_Full_B', 'Avg_Prob_Full_C', 
        'Avg_Util_A', 'Avg_Util_B', 'Avg_Util_C'
    ])

ward_a = Ward("Ward A", arrival_func=arrival_a, u=mu_a, mean_los=a_scale, std=common_std, beds=0, use_exponential=True)
ward_b = Ward("Ward B", arrival_func=arrival_b, u=mu_b, mean_los=b_scale, std=common_std, beds=0, use_exponential=True)
ward_c = Ward("Ward C", arrival_func=arrival_c, u=mu_c, mean_los=c_scale, std=common_std, beds=0, use_exponential=True)

scenarios = [25,50, 75, 100, 125]

for total_capacity in scenarios:
    print(f"\n--- RUNNING SCENARIO: {total_capacity} TOTAL BEDS ---")
    dist = get_all_bed_distributions(total_beds=total_capacity)
    
    optimal_beds, min_relocated = find_optimal_distribution(dist, ward_a, ward_b, ward_c, replications=5, csv_filename=csv_file)

    print(f"\nOptimal Configuration for {total_capacity} beds:")
    print(f"Ward A: {optimal_beds[1]}, Ward B: {optimal_beds[2]}, Ward C: {optimal_beds[3]}")
    print(f"Minimum Average Relocated Patients: {min_relocated:.2f}")
    print("-" * 40)